
# System Demonstration — Deployed AutoML Intrusion Detection System

## Objective

This notebook demonstrates the deployment and real-time operation of the AutoML-based Intrusion Detection System (IDS).

The deployed system integrates:

• FastAPI inference backend  
• H2O AutoML MOJO model  
• Automated preprocessing and feature alignment  
• Streamlit dashboard user interface  

This confirms that the proposed AutoML IDS framework can operate in real-time environments.


---
## System Architecture Overview

The deployed IDS consists of three layers:

### 1. User Interface Layer (Streamlit)
Allows users to upload network flow data and view predictions.

### 2. Inference Layer (FastAPI)
Handles incoming prediction requests and runs preprocessing and model inference.

### 3. Model Layer (H2O AutoML MOJO)
Performs intrusion detection using the trained AutoML-selected model.

Pipeline flow:

Raw Network Data → Preprocessor → Feature Alignment → MOJO Model → Prediction → Dashboard


In [1]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
DEPLOY_DIR = BASE_DIR / "deployment"
ART_DIR = DEPLOY_DIR / "artifacts"

files = list(ART_DIR.glob("*"))

print("Deployment artifacts:")
for f in files:
    print("✓", f.name)


Deployment artifacts:
✓ final_feature_names.json
✓ GBM_1_AutoML_1_20260213_181254.zip
✓ h2o-genmodel.jar
✓ preprocessor.joblib
✓ raw_schema.json
✓ selected_features.json
✓ threshold.json
✓ _tmp_input.csv
✓ _tmp_output.csv


---
## API Connectivity Test

We verify that the FastAPI deployment server is operational.


In [4]:
import requests

url = "http://127.0.0.1:8000/health"

response = requests.get(url)

print("Status code:", response.status_code)
print("Response:", response.json())


Status code: 200
Response: {'status': 'ok'}


In [5]:
import requests

API_URL = "http://127.0.0.1:8000"

try:
    r = requests.get(f"{API_URL}/health", timeout=5)
    print("Status code:", r.status_code)
    print("Text:", r.text)
    if r.headers.get("content-type","").startswith("application/json"):
        print("JSON:", r.json())
except Exception as e:
    print("❌ Health check failed.")
    print("Reason:", repr(e))
    print("\n✅ Fix checklist:")
    print("1) Is FastAPI running? Start it with:")
    print("   cd deployment/app")
    print("   uvicorn main:app --reload")
    print("2) Is it running on port 8000? If not, change API_URL above.")
    print("3) If you see 'No module named requests', install it:")
    print("   pip install requests")


Status code: 200
Text: {"status":"ok"}
JSON: {'status': 'ok'}


---
## Single Prediction Demonstration

We send a sample network flow record to the deployed IDS and observe the prediction output.


In [6]:
payload = {
    "data": {
        "dur": 0.1,
        "sbytes": 300,
        "dbytes": 120,
        "rate": 10,
        "sttl": 254,
        "dttl": 252,
        "sload": 5.2,
        "dload": 2.1,
        "proto": "tcp",
        "service": "http",
        "state": "FIN"
    }
}

response = requests.post("http://127.0.0.1:8000/predict", json=payload)

print(response.json())


{'prob_attack': 0.892949338853851, 'threshold': 0.875211908812319, 'pred_label': 1}


---
## Batch Prediction Demonstration

We simulate real-world usage by sending multiple records to the IDS.


In [7]:
import pandas as pd

# load your UNSW testing dataset
DATA_PATH = BASE_DIR / "data"

csv_files = list(DATA_PATH.rglob("*testing*.csv"))

print("Using:", csv_files[0])

df = pd.read_csv(csv_files[0])

sample = df.sample(5, random_state=42)

results = []

for _, row in sample.iterrows():
    payload = {"data": row.to_dict()}
    r = requests.post("http://127.0.0.1:8000/predict", json=payload)
    results.append(r.json())

pd.DataFrame(results)


Using: C:\Users\sohib\Documents\Final Year Project\AutoML\data\raw\UNSW_NB15_testing-set.csv


,prob_attack,threshold,pred_label
0,0.000155,0.875212,0
1,0.998869,0.875212,1
2,0.999448,0.875212,1
3,0.000483,0.875212,0
4,0.994444,0.875212,1


---
## Deployment Validation Summary

The deployed AutoML IDS successfully:

• Accepts raw network flow input  
• Applies automated preprocessing  
• Performs inference using MOJO model  
• Returns intrusion probability and classification  

This confirms the proposed system can operate in real-time intrusion detection scenarios.


---
## Dissertation Figures (Screenshots to include)

Figure X: FastAPI Swagger interface  
Figure X: Streamlit dashboard interface  
Figure X: Single prediction output  
Figure X: Batch prediction output  

These demonstrate operational deployment.
